# QUY TRÌNH THU THẬP VÀ XỬ LÝ DỮ LIỆU NGHIÊN CỨU (CHƯƠNG 3)
Đề tài: *"How do financial inclusion and fintech impact environmental sustainability in countries committed to net-zero carbon emissions?"*

Toàn bộ code được chia theo từng phần chuẩn mực theo cấu trúc biến số tại **Mục 3.3 và Bảng 3.1**:
- **Phần 1:** Biến phụ thuộc ($CO_2$, $EF$)
- **Phần 2:** Biến độc lập chính ($FI$, $FT$)
- **Phần 3:** Biến kiểm soát ($LCE, URB, FDI, GDP, NTR, PO, TO, PS, IND, INF$)
- **Phần 4:** Biến kiểm định độ bền vững ($ED$)
- **Phần 5:** Hợp nhất toàn bộ dữ liệu thành bảng tổng thể (Master Dataset)

In [14]:

import os
import io
import requests
import pandas as pd
from functools import reduce
from dotenv import load_dotenv

load_dotenv()

os.makedirs('data', exist_ok=True)
print('Đã khởi tạo môi trường và thư mục data/ thành công!')


Đã khởi tạo môi trường và thư mục data/ thành công!


--- 
## PHẦN 1: THU THẬP BIẾN PHỤ THUỘC (DEPENDENT VARIABLES)
1. **Khí thải $CO_2$ bình quân đầu người ($CO2$):** Nguồn Our World in Data (OWID).
2. **Dấu chân sinh thái bình quân đầu người ($EF$):** Nguồn Global Footprint Network (GFN API).

In [15]:
print('Đang tải dữ liệu CO2')
co2_url = 'https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv'
co2_raw = pd.read_csv(co2_url)

# Lọc giai đoạn từ năm 2010 trở đi và các cột cần thiết
co2_df = co2_raw[co2_raw['year'] >= 2010].copy()
co2_cols = ['country', 'iso_code', 'year', 'co2_per_capita', 'co2', 'population', 'gdp']
co2_df = co2_df[[c for c in co2_cols if c in co2_df.columns]]
co2_df.rename(columns={'iso_code': 'country_code'}, inplace=True)

# Lưu file
co2_df.to_csv('data/co2_data.csv', index=False, encoding='utf-8')
print(f'-> Đã lưu dữ liệu CO2 ({len(co2_df)} dòng) vào: data/co2_data.csv')

print('\n Đang tải dữ liệu Ecological Footprint (EF)')
gfn_key = os.getenv('Global_Footprint_Key')
ef_api_url = 'https://api.footprintnetwork.org/v1/data/all/all/EFCpc'
gfn_headers = {'HTTP_ACCEPT': 'application/json'}

ef_res = requests.get(ef_api_url, auth=('username', gfn_key), headers=gfn_headers)
if ef_res.status_code == 200:
    ef_json = ef_res.json()
    ef_df = pd.DataFrame(ef_json)
    ef_df = ef_df[ef_df['year'] >= 2010].copy()
    ef_df.rename(columns={'value': 'ef_per_capita'}, inplace=True)
    
    ef_cols = ['countryName', 'shortName', 'isoa2', 'year', 'record', 'ef_per_capita', 
               'cropLand', 'grazingLand', 'forestLand', 'fishingGround', 'builtupLand', 'carbon']
    ef_df = ef_df[[c for c in ef_cols if c in ef_df.columns]]
    
    ef_df.to_csv('data/ef_data.csv', index=False, encoding='utf-8')
    print(f'-> Đã lưu dữ liệu EF ({len(ef_df)} dòng) vào: data/ef_data.csv')
else:
    print(f'Lỗi tải GFN API: {ef_res.status_code}')


Đang tải dữ liệu CO2
-> Đã lưu dữ liệu CO2 (3810 dòng) vào: data/co2_data.csv

 Đang tải dữ liệu Ecological Footprint (EF)


KeyboardInterrupt: 

--- 
## PHẦN 2: THU THẬP BIẾN ĐỘC LẬP CHÍNH (INDEPENDENT VARIABLES)
1. **Bao trùm Tài chính ($FI$):** Gồm 4 chỉ báo ($BANK, ATM, DEP, LOAN$) từ IMF Financial Access Survey (FAS).
2. **Công nghệ Tài chính ($FT$):** Gồm 3 chỉ báo (Thuê bao di động, Băng rộng cố định, Tỷ lệ dùng Internet) từ World Bank WDI.

In [ ]:

print(' Đang xử lý chỉ báo Bao trùm Tài chính (FI) ')
fas_raw_path = 'data/imf_fas_data.csv'

if not os.path.exists(fas_raw_path):
    print('Đang tải dataset IMF FAS từ API SDMX 3.0...')
    fas_url = 'https://api.imf.org/external/sdmx/3.0/data/dataflow/IMF.STA/FAS/~/*?startPeriod=2010&endPeriod=2025'
    r_fas = requests.get(fas_url, headers={'Accept': 'text/csv'})
    with open(fas_raw_path, 'w', encoding='utf-8') as f:
        f.write(r_fas.text)

fas_cols = ['COUNTRY', 'INDICATOR', 'TYPE_OF_TRANSFORMATION', 'TIME_PERIOD', 'OBS_VALUE']
raw_fas = pd.read_csv(fas_raw_path, usecols=fas_cols, low_memory=False)
raw_fas = raw_fas[raw_fas['TIME_PERIOD'] >= 2010].copy()

bank_df = raw_fas[(raw_fas['INDICATOR'] == 'FA26N') & (raw_fas['TYPE_OF_TRANSFORMATION'] == 'PHTADLT_NUM')][['COUNTRY', 'TIME_PERIOD', 'OBS_VALUE']].rename(columns={'OBS_VALUE': 'BANK'})
atm_df = raw_fas[(raw_fas['INDICATOR'] == 'FA18N') & (raw_fas['TYPE_OF_TRANSFORMATION'] == 'PHTADLT_NUM')][['COUNTRY', 'TIME_PERIOD', 'OBS_VALUE']].rename(columns={'OBS_VALUE': 'ATM'})
dep_df = raw_fas[(raw_fas['INDICATOR'] == 'OUTD_COMBANK') & (raw_fas['TYPE_OF_TRANSFORMATION'] == 'POGDP')][['COUNTRY', 'TIME_PERIOD', 'OBS_VALUE']].rename(columns={'OBS_VALUE': 'DEP'})
loan_df = raw_fas[(raw_fas['INDICATOR'] == 'OUTL_COMBANK') & (raw_fas['TYPE_OF_TRANSFORMATION'] == 'POGDP')][['COUNTRY', 'TIME_PERIOD', 'OBS_VALUE']].rename(columns={'OBS_VALUE': 'LOAN'})

fi_df = reduce(lambda l, r: pd.merge(l, r, on=['COUNTRY', 'TIME_PERIOD'], how='outer'), [bank_df, atm_df, dep_df, loan_df])
fi_df.rename(columns={'COUNTRY': 'country_code', 'TIME_PERIOD': 'year'}, inplace=True)
fi_df['year'] = fi_df['year'].astype(int)
fi_df.sort_values(by=['country_code', 'year'], inplace=True)
fi_df.to_csv('data/fi_data.csv', index=False, encoding='utf-8')
print(f'-> Đã lưu dữ liệu FI ({len(fi_df)} dòng) vào: data/fi_data.csv')

print('\n Đang tải 3 chỉ báo Fintech (FT) ')
ft_indicators = {
    'FT_Mobile': 'IT.CEL.SETS.P2',       # Thuê bao di động trên 100 người
    'FT_Broadband': 'IT.NET.BBND.P2',   # Thuê bao băng rộng trên 100 người
    'FT_Internet': 'IT.NET.USER.ZS'     # Tỷ lệ người dân dùng Internet (% dân số)
}

ft_list = []
for var_name, code in ft_indicators.items():
    url = f'https://api.worldbank.org/v2/country/all/indicator/{code}?format=json&date=2010:2025&per_page=20000'
    res = requests.get(url).json()
    if len(res) > 1 and res[1]:
        data = [{'country_code': r['countryiso3code'], 'country_name': r['country']['value'], 'year': int(r['date']), var_name: r['value']} for r in res[1] if r['countryiso3code']]
        ft_list.append(pd.DataFrame(data))

ft_df = reduce(lambda l, r: pd.merge(l, r, on=['country_code', 'country_name', 'year'], how='outer'), ft_list)
ft_df.sort_values(by=['country_code', 'year'], inplace=True)
ft_df.to_csv('data/ft_data.csv', index=False, encoding='utf-8')
print(f'-> Đã lưu dữ liệu FT ({len(ft_df)} dòng) vào: data/ft_data.csv')


 Đang xử lý chỉ báo Bao trùm Tài chính (FI) 
-> Đã lưu dữ liệu FI (2764 dòng) vào: data/fi_data.csv

 Đang tải 3 chỉ báo Fintech (FT) 
-> Đã lưu dữ liệu FT (4160 dòng) vào: data/ft_data.csv


--- 
## PHẦN 3: THU THẬP BIẾN KIỂM SOÁT (CONTROL VARIABLES)
Bao gồm các biến số kinh tế - xã hội - thể chế trích xuất từ World Bank WDI & Worldwide Governance Indicators (WGI):
- $LCE$: Năng lượng các-bon thấp (`EG.FEC.RNEW.ZS`)
- $URB$: Tỷ lệ đô thị hóa (`SP.URB.TOTL.IN.ZS`)
- $FDI$: Dòng vốn đầu tư FDI ròng (% GDP) (`BX.KLT.DINV.WD.GD.ZS`)
- $GDP$: GDP thực bình quân đầu người (`NY.GDP.PCAP.KD`)
- $NTR$: Thu nhập từ tài nguyên thiên nhiên (% GDP) (`NY.GDP.TOTL.RT.ZS`)
- $PO$: Quy mô dân số (`SP.POP.TOTL`)
- $TO$: Độ mở thương mại (% GDP) (`NE.TRD.GNFS.ZS`)
- $IND$: Tỷ trọng công nghiệp (% GDP) (`NV.IND.TOTL.ZS`)
- $INF$: Tỷ lệ lạm phát (Chỉ số giảm phát GDP) (`NY.GDP.DEFL.KD.ZG`)
- $PS$: Điểm ổn định chính trị từ WGI (`GOV_WGI_PV.EST`)

In [ ]:
print(' Đang tải các biến kiểm soát từ World Bank API ')
ctrl_indicators = {
    'LCE': 'EG.FEC.RNEW.ZS',
    'URB': 'SP.URB.TOTL.IN.ZS',
    'FDI': 'BX.KLT.DINV.WD.GD.ZS',
    'GDP': 'NY.GDP.PCAP.KD',
    'NTR': 'NY.GDP.TOTL.RT.ZS',
    'PO': 'SP.POP.TOTL',
    'TO': 'NE.TRD.GNFS.ZS',
    'IND': 'NV.IND.TOTL.ZS',
    'INF': 'NY.GDP.DEFL.KD.ZG'
}

ctrl_list = []
for var_name, code in ctrl_indicators.items():
    url = f'https://api.worldbank.org/v2/country/all/indicator/{code}?format=json&date=2010:2025&per_page=20000'
    res = requests.get(url).json()
    if len(res) > 1 and res[1]:
        data = [{'country_code': r['countryiso3code'], 'country_name': r['country']['value'], 'year': int(r['date']), var_name: r['value']} for r in res[1] if r['countryiso3code']]
        ctrl_list.append(pd.DataFrame(data))

# Tải thêm biến Ổn định chính trị (PS) từ WGI (source=3)
ps_url = 'https://api.worldbank.org/v2/country/all/indicator/GOV_WGI_PV.EST?format=json&date=2010:2025&source=3&per_page=20000'
res_ps = requests.get(ps_url).json()
if len(res_ps) > 1 and res_ps[1]:
    ps_data = [{'country_code': r['countryiso3code'], 'country_name': r['country']['value'], 'year': int(r['date']), 'PS': r['value']} for r in res_ps[1] if r['countryiso3code']]
    ctrl_list.append(pd.DataFrame(ps_data))

ctrl_df = reduce(lambda l, r: pd.merge(l, r, on=['country_code', 'country_name', 'year'], how='outer'), ctrl_list)
ctrl_df.sort_values(by=['country_code', 'year'], inplace=True)
ctrl_df.to_csv('data/control_data.csv', index=False, encoding='utf-8')
print(f'-> Đã lưu các biến kiểm soát ({len(ctrl_df)} dòng) vào: data/control_data.csv')


 Đang tải các biến kiểm soát từ World Bank API 
-> Đã lưu các biến kiểm soát (4175 dòng) vào: data/control_data.csv


--- 
## PHẦN 4: THU THẬP BIẾN KIỂM ĐỊNH ĐỘ BỀN VỮNG (ROBUSTNESS TEST VARIABLES)
- $ED$: Khí thải metan nông nghiệp (Triệu tấn $CO_2$ tương đương) từ World Bank WDI (`EN.GHG.CH4.AG.MT.CE.AR5`).

In [ ]:
print('Đang tải biến kiểm định độ bền vững (ED)')
ed_code = 'EN.GHG.CH4.AG.MT.CE.AR5'
url = f'https://api.worldbank.org/v2/country/all/indicator/{ed_code}?format=json&date=2010:2025&per_page=20000'
res = requests.get(url).json()

if len(res) > 1 and res[1]:
    data = [{'country_code': r['countryiso3code'], 'country_name': r['country']['value'], 'year': int(r['date']), 'ED': r['value']} for r in res[1] if r['countryiso3code']]
    ed_df = pd.DataFrame(data)
    ed_df.sort_values(by=['country_code', 'year'], inplace=True)
    ed_df.to_csv('data/robustness_ed_data.csv', index=False, encoding='utf-8')
    print(f'-> Đã lưu biến Robustness ED ({len(ed_df)} dòng) vào: data/robustness_ed_data.csv')


Đang tải biến kiểm định độ bền vững (ED)
-> Đã lưu biến Robustness ED (4160 dòng) vào: data/robustness_ed_data.csv


--- 
## PHẦN 5: HỢP NHẤT TOÀN BỘ BẢNG DỮ LIỆU (MASTER DATASET)
Hợp nhất các bảng dữ liệu thành một tập dữ liệu bảng (Panel Data) hoàn chỉnh theo cặp định danh `country_code` (ISO3) và `year` cho giai đoạn **2011 – 2023**.

In [16]:
print('Đang tiến hành hợp nhất toàn bộ dữ liệu ')

# Đọc lại các file dữ liệu thành phần
co2_df = pd.read_csv('data/co2_data.csv')
fi_df = pd.read_csv('data/fi_data.csv')
ft_df = pd.read_csv('data/ft_data.csv')
ctrl_df = pd.read_csv('data/control_data.csv')
ed_df = pd.read_csv('data/robustness_ed_data.csv')

# Chuẩn hóa dữ liệu EF (ghép country_code ISO3 từ bảng CO2 hoặc WDI)
ef_raw = pd.read_csv('data/ef_data.csv')
country_lookup = ctrl_df[['country_code', 'country_name']].drop_duplicates()
ef_merged = pd.merge(ef_raw, country_lookup, left_on='countryName', right_on='country_name', how='left')
ef_clean = ef_merged[['country_code', 'year', 'ef_per_capita', "cropLand", 'grazingLand', 'forestLand' ,'fishingGround' ,'builtupLand' ,'carbon']].dropna(subset=['country_code'])

# Danh sách các bảng cần gộp
dfs_to_merge = [
    co2_df[['country_code', 'country', 'year', 'co2_per_capita', 'co2']],
    ef_clean,
    fi_df,
    ft_df[['country_code', 'year', 'FT_Mobile', 'FT_Broadband', 'FT_Internet']],
    ctrl_df[['country_code', 'year', 'LCE', 'URB', 'FDI', 'GDP', 'NTR', 'PO', 'TO', 'IND', 'INF', 'PS']],
    ed_df[['country_code', 'year', 'ED']]
]

# Hợp nhất theo country_code và year
master_df = reduce(lambda left, right: pd.merge(left, right, on=['country_code', 'year'], how='outer'), dfs_to_merge)

# Lọc chuẩn giai đoạn nghiên cứu từ 2011 đến 2023
mask = (master_df['year'] >= 2010) & (master_df['year'] <= 2025)
master_df = master_df[mask].copy()
master_df.sort_values(by=['country_code', 'year'], inplace=True)

# Lưu file Master Dataset
master_df.to_csv('data/raw_master_dataset.csv', index=False, encoding='utf-8-sig')
print(f'-> ĐÃ HỢP NHẤT THÀNH CÔNG MASTER DATASET: {master_df.shape[0]} dòng x {master_df.shape[1]} cột!')
print('File lưu tại: data/raw_master_dataset.csv\n')

# Hiển thị 5 dòng đầu
print(master_df.head(5))


Đang tiến hành hợp nhất toàn bộ dữ liệu 
-> ĐÃ HỢP NHẤT THÀNH CÔNG MASTER DATASET: 4926 dòng x 30 cột!
File lưu tại: data/raw_master_dataset.csv

  country_code country  year  co2_per_capita    co2  ef_per_capita  cropLand  \
0          ABW   Aruba  2010          25.030  2.506            NaN       NaN   
1          ABW   Aruba  2011          24.706  2.495            NaN       NaN   
2          ABW   Aruba  2012          13.212  1.345            NaN       NaN   
3          ABW   Aruba  2013           8.395  0.861            NaN       NaN   
4          ABW   Aruba  2014           8.435  0.872            NaN       NaN   

   grazingLand  forestLand  fishingGround  ...        URB        FDI  \
0          NaN         NaN            NaN  ...  64.161297   7.611672   
1          NaN         NaN            NaN  ...  63.999261  18.505780   
2          NaN         NaN            NaN  ...  63.825209 -12.033180   
3          NaN         NaN            NaN  ...  63.638291   9.974457   
4          Na

In [ ]:
df = pd.read_csv('data/master_dataset.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4926 entries, 0 to 4925
Data columns (total 30 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country_code    4386 non-null   object 
 1   country         3825 non-null   object 
 2   year            4926 non-null   int64  
 3   co2_per_capita  3480 non-null   float64
 4   co2             3720 non-null   float64
 5   ef_per_capita   1774 non-null   float64
 6   cropLand        1134 non-null   float64
 7   grazingLand     1134 non-null   float64
 8   forestLand      1134 non-null   float64
 9   fishingGround   1134 non-null   float64
 10  builtupLand     1134 non-null   float64
 11  carbon          1134 non-null   float64
 12  BANK            2702 non-null   float64
 13  ATM             2670 non-null   float64
 14  DEP             2618 non-null   float64
 15  LOAN            2622 non-null   float64
 16  FT_Mobile       3635 non-null   float64
 17  FT_Broadband    3540 non-null   f

In [ ]:
df.describe()

,year,co2_per_capita,co2,ef_per_capita,cropLand,grazingLand,forestLand,fishingGround,builtupLand,carbon,...,URB,FDI,GDP,NTR,PO,TO,IND,INF,PS,ED
count,4926.000000,3480.000000,3720.000000,1774.000000,1134.000000,1134.000000,1134.000000,1134.000000,1134.000000,1134.000000,...,4160.000000,3687.000000,3996.000000,3007.000000,4.160000e+03,3465.000000,3837.000000,4000.000000,3087.000000,3672.000000
mean,2017.423873,4.972782,932.062517,3.058113,0.578857,0.298482,0.470448,0.118990,0.085300,1.573888,...,60.036095,7.370909,15555.858394,6.042942,2.836733e+08,86.844179,26.066983,69.801259,-0.011522,161.450245
std,4.570907,5.843551,3676.396442,2.345742,0.333373,0.501881,0.526134,0.148525,0.054776,1.656217,...,22.622741,58.935809,22535.063160,9.047058,9.426432e+08,55.305069,11.537434,3584.761571,0.988212,508.361268
min,2010.000000,0.022000,0.000000,0.272142,0.078431,0.003512,0.011397,0.000412,0.004143,0.026320,...,10.984310,-1303.108267,253.308578,0.000000,9.492000e+03,0.000000,0.000000,-42.918604,-3.016512,0.000000
25%,2013.000000,0.977750,1.693750,1.262896,0.322942,0.082232,0.174999,0.028720,0.039431,0.251624,...,41.733696,1.222373,2108.270825,0.261915,1.502597e+06,52.022995,18.608987,1.550035,-0.650647,0.572950
50%,2017.000000,3.360000,13.419500,2.386854,0.499738,0.183519,0.272099,0.072214,0.077855,1.103543,...,60.369000,2.511559,6139.726300,2.201002,1.007072e+07,73.537288,25.039083,3.459072,0.031523,5.282750
75%,2021.000000,6.710750,150.618750,4.256564,0.751530,0.356773,0.572798,0.139832,0.116995,2.467201,...,78.555595,4.625017,18982.361768,7.885171,5.175450e+07,105.130362,31.846858,6.444630,0.842078,39.797300
max,2025.000000,49.152000,38598.578000,17.643144,1.939317,5.067615,3.918504,1.159158,0.277943,12.599307,...,100.000000,1709.827232,247170.103807,79.430949,8.215425e+09,679.232773,78.899528,225652.234994,1.759751,4333.608958
